# 016 — Public-supervised D/LIVE with broadcast and tactical features

Отдельная проверка новой информации: 92 тактических признака + доступность broadcast, отсутствие гомографии, слабое правило D, время от начала/до конца отсутствия гомографии, вероятности 009 и 010. Родительские модели обучены вне публичного окна, публичные метки в их веса не входили. Признаки broadcast используют офлайн-синхронизацию и будущий контекст, не являются эталоном.

Обучение только на публичном префиксе: подбор на четвёртой минуте, обучение на первых трёх с 20-секундным отступом; после выбора обучение на первых четырёх с тем же отступом перед последней минутой. Всего одна настоящая остановка во внутреннем обучении и две в финальном. Последняя минута многократно исследовалась, не независимый тест. Пять минут смешивают обучение, подбор и диагностику.

Шесть конфигураций: логрегрессия (2), бустинг (2), лес, небольшая MLP. Порог и длительность выбираются при precision .95/.99 на четвёртой минуте, минимум 100 D-кадров. Если требование не выполнено, D не выдаётся. Это отдельный эксперимент с добавленной информацией, не чистое сравнение архитектур с 010–013.

Run All после 009/010 и broadcast_alignment. Выходы только local_review/experiments/016_dead_broadcast_features. tqdm и progress.json. Все расчёты в ноутбуке; одобренные библиотеки. Публикуются код, конфигурация, обученная модель, но не данные или прогнозы. Все функции для восстановления вероятностей из доверенных собственных pickle есть в ноутбуке. Требуются родительские модели 009/010. H/A не назначаются, spellF1 неприменима.


In [ ]:
EXPERIMENT='016_dead_broadcast_features'
CONFIGS=[{'name': 'logistic_c01', 'kind': 'logistic', 'C': 0.1}, {'name': 'logistic_c1', 'kind': 'logistic', 'C': 1.0}, {'name': 'boost_shallow', 'kind': 'boost', 'depth': 2, 'leaves': 4, 'leaf': 15, 'trees': 150, 'l2': 5.0}, {'name': 'boost_medium', 'kind': 'boost', 'depth': 3, 'leaves': 7, 'leaf': 10, 'trees': 150, 'l2': 5.0}, {'name': 'forest', 'kind': 'rf', 'depth': 6, 'leaf': 10}, {'name': 'mlp_small', 'kind': 'mlp', 'hidden': [32], 'dropout': 0.2, 'weight_decay': 0.05, 'epochs': 40}]


In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
from pathlib import Path
import json, sys, pickle, time, hashlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
import torch
from torch import nn
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
sys.path.insert(0,str(ROOT))
from games import GAME,half_video_blocks
OUT=ROOT/'local_review/experiments'/EXPERIMENT
OUT.mkdir(parents=True,exist_ok=True)
FPS=25; N=199319; START=11185; END=18685; GUARD=500
BLOCKS=half_video_blocks(GAME)
FEATURE_PATH=ROOT/'local_review/experiments/008_weak_dead_boosting/features.npy'
X=np.load(FEATURE_PATH,mmap_mode='r')
feature_names=json.loads(FEATURE_PATH.with_name('feature_names.json').read_text())
align=ROOT/'local_review/broadcast_alignment'
weak_d=np.load(align/'training_weak_D_mask.npy')
weak_live=np.load(align/'training_weak_LIVE_mask.npy')
assert not (weak_d&weak_live).any()
assert not (weak_d|weak_live)[START-GUARD:END+GUARD].any()
weak_y=np.full(N,-1,np.int8);weak_y[weak_live]=0;weak_y[weak_d]=1
# Public labels are permitted. Only the explicitly selected prefix enters training/selection.
# Публичные метки разрешены; в обучение и подбор входит только явно заданный префикс.
gt=pd.read_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')
assert np.array_equal(gt.frame,np.arange(N))
truth=gt.label.eq('D').to_numpy();public=gt.label.isin(['H','A','D']).to_numpy()
assert public.sum()==7500
positions=np.arange(N)
graded=np.zeros(N,bool)
for _,a,b in BLOCKS:graded[a:b+1]=True
THRESHOLDS=[.1,.2,.3,.4,.5,.6,.7,.75,.8,.85,.9,.925,.95,.97,.98,.99,.995,.999]
MIN_DURATIONS=[0,25,50,100]
TARGETS=[.95,.99]
MIN_PREDICTED=100 # 4 seconds / Минимум 4 секунды предсказанного D при подборе.
def save_json(path,obj):
 tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2));tmp.replace(path)
def progress(**kw):save_json(OUT/'progress.json',kw)
def spans(mask):
 edges=np.r_[0,np.flatnonzero(mask[1:]!=mask[:-1])+1,len(mask)]
 return zip(edges[:-1],edges[1:])
def filter_segment(raw,min_duration):
 if min_duration==0:return raw.copy()
 # Offline filter: bridge <=0.4s, then remove short D. Never crosses a half boundary.
 # Офлайн-фильтр: закрываем разрывы до 0,4 с, затем удаляем короткие D; таймы не соединяем.
 out=raw.copy()
 for a,b in spans(raw):
  if not raw[a] and a>0 and b<len(raw) and b-a<=10:out[a:b]=True
 for a,b in list(spans(out)):
  if out[a] and b-a<min_duration:out[a:b]=False
 return out

def score(y,p):
 y=np.asarray(y,bool);p=np.asarray(p,bool)
 tp=int((y&p).sum());fp=int((~y&p).sum());fn=int((y&~p).sum());tn=int((~y&~p).sum())
 f1=2*tp/max(2*tp+fp+fn,1);lf1=2*tn/max(2*tn+fp+fn,1)
 return dict(D_precision=tp/max(tp+fp,1),D_recall=tp/max(tp+fn,1),D_F1=f1,
             LIVE_F1=lf1,binary_macroF1=(f1+lf1)/2,true_D_frames=tp+fn,predicted_D_frames=tp+fp,
             true_positive=tp,false_positive=fp,false_negative=fn)

def make_training(regime,final=False):
 # Weak-only development uses half 1, weak holdout half 2. Final uses both halves.
 # Слабое обучение: подбор на втором тайме, итоговое обучение на обоих таймах.
 weak_mask=weak_y>=0
 if regime=='weak' and not final:weak_mask &= positions<BLOCKS[1][1]
 ids=[];labels=[];weights=[]
 for label in [0,1]:
  mask=weak_mask&(weak_y==label)
  for a,b in spans(mask):
   if not mask[a]:continue
   chosen=np.arange(a,b,5)
   if len(chosen)>50:chosen=chosen[np.linspace(0,len(chosen)-1,50).astype(int)]
   ids.extend(chosen);labels.extend([label]*len(chosen));weights.extend([1/len(chosen)]*len(chosen))
 ids=np.array(ids,int);labels=np.array(labels,int);weights=np.array(weights,float)
 for label in [0,1]:weights[labels==label]/=weights[labels==label].sum()
 weights*=.5
 if regime in ['hybrid','public']:
  # Inner validation = minute 4; final diagnostic = minute 5; 20s guard before each.
  # Внутренний подбор — минута 4; итоговая диагностика — минута 5; перед каждой отступ 20 с.
  limit=START+(6000 if final else 4500)-GUARD
  labeled_ids=np.arange(START,limit,5)
  assert public[labeled_ids].all()
  labeled_y=truth[labeled_ids].astype(int);w=np.ones(len(labeled_ids),float)
  for label in [0,1]:
   assert (labeled_y==label).any()
   w[labeled_y==label]/=w[labeled_y==label].sum()
  # Most total weight comes from trustworthy public labels, with balanced classes.
  # Большая часть веса у достоверных публичных меток; классы сбалансированы.
  weights*=.25;w*=.75*.5
  if regime=='public':
   ids=np.array([],dtype=int);labels=np.array([],dtype=int);weights=np.array([],dtype=float);w=w/(w.sum())
  ids=np.r_[ids,labeled_ids];labels=np.r_[labels,labeled_y];weights=np.r_[weights,w]
 weights*=len(weights)/weights.sum()
 assert not ((ids>=START+6000)&(ids<END)).any()
 return ids,labels,weights

def prep():
 # Imputation statistics learned ONLY on training data; missingness is explicit.
 # Заполнение пропусков и масштабирование обучаются ТОЛЬКО на обучающей выборке.
 return make_pipeline(SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True),StandardScaler())

def network(n_input,hidden,dropout):
 layers=[];previous=n_input
 for size in hidden:layers.extend([nn.Linear(previous,size),nn.ReLU(),nn.Dropout(dropout)]);previous=size
 layers.append(nn.Linear(previous,1))
 return nn.Sequential(*layers)

def fit_variant(config,ids,y,w,tag):
 seed=42;torch.manual_seed(seed);np.random.seed(seed);torch.set_num_threads(4)
 kind=config['kind']
 if kind=='logistic':
  pre=prep();z=pre.fit_transform(X[ids])
  model=LogisticRegression(C=config['C'],max_iter=3000,solver='lbfgs',random_state=seed)
  model.fit(z,y,sample_weight=w)
  return dict(kind=kind,preprocessor=pre,model=model)
 if kind in ['rf','extra']:
  cls=RandomForestClassifier if kind=='rf' else ExtraTreesClassifier
  model=cls(n_estimators=200,max_depth=config['depth'],min_samples_leaf=config['leaf'],max_features=.7,n_jobs=4,random_state=seed)
  model.fit(X[ids],y,sample_weight=w)
  return dict(kind=kind,model=model)
 if kind=='boost':
  model=HistGradientBoostingClassifier(max_iter=config['trees'],learning_rate=.05,max_depth=config['depth'],max_leaf_nodes=config['leaves'],min_samples_leaf=config['leaf'],l2_regularization=config['l2'],early_stopping=False,random_state=seed)
  model.fit(X[ids],y,sample_weight=w)
  return dict(kind=kind,model=model)
 pre=prep();z=np.clip(pre.fit_transform(X[ids]),-10,10).astype(np.float32)
 net=network(z.shape[1],config['hidden'],config['dropout'])
 optimizer=torch.optim.AdamW(net.parameters(),lr=.001,weight_decay=config['weight_decay'])
 xx=torch.from_numpy(z);yy=torch.tensor(y,dtype=torch.float32);ww=torch.tensor(w,dtype=torch.float32)
 generator=torch.Generator().manual_seed(seed);losses=[]
 for epoch in tqdm(range(config['epochs']),desc=tag,leave=False):
  net.train();order=torch.randperm(len(y),generator=generator);total=0.
  for chunk in order.split(256):
   optimizer.zero_grad();logits=net(xx[chunk]).squeeze(1)
   loss=(nn.functional.binary_cross_entropy_with_logits(logits,yy[chunk],reduction='none')*ww[chunk]).mean()
   loss.backward();optimizer.step();total+=float(loss.detach())*len(chunk)
  losses.append(total/len(y));progress(stage=tag,epoch=epoch+1,total_epochs=config['epochs'],loss=losses[-1])
 return dict(kind='mlp',preprocessor=pre,hidden=config['hidden'],dropout=config['dropout'],n_input=z.shape[1],
             state={k:v.detach().numpy() for k,v in net.state_dict().items()},losses=losses)

def predict(bundle,ids):
 result=np.empty(len(ids),np.float32)
 if bundle['kind']=='mlp':
  net=network(bundle['n_input'],bundle['hidden'],bundle['dropout'])
  net.load_state_dict({k:torch.from_numpy(v) for k,v in bundle['state'].items()});net.eval()
 for a in range(0,len(ids),8192):
  z=X[ids[a:a+8192]]
  if 'preprocessor' in bundle:z=bundle['preprocessor'].transform(z)
  if bundle['kind']=='mlp':
   with torch.no_grad():p=net(torch.tensor(np.clip(z,-10,10),dtype=torch.float32)).squeeze(1).sigmoid().numpy()
  else:p=bundle['model'].predict_proba(z)[:,1]
  result[a:a+len(p)]=p
 assert np.isfinite(result).all()
 return result

def choose_settings(prob,y,mask):
 records=[]
 for threshold in THRESHOLDS:
  for duration in MIN_DURATIONS:
   pred=filter_segment(prob>=threshold,duration)
   row=score(y[mask],pred[mask]);row.update(threshold=threshold,min_dead_frames=duration)
   records.append(row)
 choices={}
 for target in TARGETS:
  feasible=[r for r in records if r['D_precision']>=target and r['predicted_D_frames']>=MIN_PREDICTED]
  if feasible:
   best=max(feasible,key=lambda r:(r['D_recall'],r['D_precision'],-r['min_dead_frames']))
   choices[str(target)]={**best,'feasible':True}
  else:
   # Explicit abstention from D if the target is unreachable; no fake perfect precision.
   # Если целевая точность недостижима, не объявляем D и не называем precision идеальным.
   choices[str(target)]=dict(threshold=1.1,min_dead_frames=25,feasible=False,D_precision=0.,D_recall=0.,predicted_D_frames=0)
 return choices,records

# All added signals are automatic, never private/reference labels.
# Все дополнительные сигналы автоматические, без закрытых или эталонных меток.
base_features=X
mapping=pd.read_csv(align/'broadcast_to_tactical.csv')
homography=np.load(align/'broadcast_sampled_geometry.npz')['homography_valid']
known=np.zeros(N,bool);invalid=np.full(N,np.nan,np.float32)
valid_rows=mapping.tactical_frame.to_numpy()>=0
mapped_frames=mapping.tactical_frame.to_numpy()[valid_rows].astype(int)
known[mapped_frames]=True;invalid[mapped_frames]=(~homography[valid_rows]).astype(np.float32)
elapsed=np.full(N,np.nan,np.float32);remaining=elapsed.copy()
elapsed[known]=0.;remaining[known]=0.
for a,b in spans(invalid==1):
 if invalid[a]!=1:continue
 elapsed[a:b]=np.arange(b-a)/25;remaining[a:b]=np.arange(b-a-1,-1,-1)/25
parents=[]
for parent in ['009_broadcast_weak_boosting','010_dead_logistic']:
 local=ROOT/'local_review/experiments'/parent
 probability_path=local/('p_D.npy' if parent.startswith('009') else 'weak_p_D.npy')
 if probability_path.exists():p=np.load(probability_path)
 else:
  model_path=ROOT/'sol'/parent/('model.pkl' if parent.startswith('009') else 'weak_model.pkl')
  with model_path.open('rb') as f:bundle=pickle.load(f)
  if parent.startswith('009'):bundle=dict(kind='boost',model=bundle)
  p=predict(bundle,np.arange(N))
 parents.append(p)
extra=np.column_stack([known,invalid,np.where(known,np.load(align/'weak_D_candidates.npy'),np.nan),elapsed,remaining,*parents])
X=np.column_stack([base_features,extra]).astype(np.float32)
feature_names=feature_names+['broadcast_mapped','broadcast_invalid','broadcast_weak_D','invalid_elapsed_s','invalid_remaining_s','probability_009','probability_010']
assert X.shape==(N,len(feature_names))


In [ ]:
all_results={};search_rows=[]
for regime in ['public']:
 train_ids,y,w=make_training(regime)
 if regime=='weak':
  a,b=BLOCKS[1][1],BLOCKS[1][2]+1
  val_ids=np.arange(a,b);val_mask=weak_y[val_ids]>=0;val_y=weak_y[val_ids]==1
 else:
  val_ids=np.arange(START+4500-250,START+6000+250)
  # Include unlabelled context for the offline filter; score ONLY minute four.
  # Контекст без меток нужен офлайн-фильтру; оцениваем ТОЛЬКО четвёртую минуту.
  val_mask=(val_ids>=START+4500)&(val_ids<START+6000);val_y=truth[val_ids]
 assert not np.intersect1d(train_ids,val_ids[val_mask]).size
 candidates=[]
 for config in tqdm(CONFIGS,desc=f'{EXPERIMENT} {regime} candidates'):
  progress(stage='candidate',regime=regime,config=config)
  fitted=fit_variant(config,train_ids,y,w,f'{regime} {config["name"]}')
  prob=predict(fitted,val_ids)
  choices,rows=choose_settings(prob,val_y,val_mask)
  ap=float(average_precision_score(val_y[val_mask],prob[val_mask]))
  for row in rows:search_rows.append(dict(regime=regime,variant=config['name'],**row))
  candidates.append(dict(config=config,choices=choices,average_precision=ap))
  # Local development probabilities permit auditing the selection without refitting.
  # Вероятности подбора сохраняем локально для воспроизводимой проверки.
  np.save(OUT/f'{regime}_{config["name"]}_validation_probability.npy',prob)
 # Select the family winner by validation recall subject to P>=0.95; AP breaks ties.
 # Выбор внутри семейства: recall при precision>=0.95; AP разрешает равенство.
 winner=max(candidates,key=lambda c:(c['choices']['0.95']['feasible'],c['choices']['0.95']['D_recall'],c['average_precision']))
 final_ids,final_y,final_w=make_training(regime,final=True)
 progress(stage='final_fit',regime=regime,variant=winner['config']['name'])
 final_model=fit_variant(winner['config'],final_ids,final_y,final_w,f'{regime} final')
 # Same estimator is refit after validation; thresholds are transferred, NOT re-tuned on test.
 # Модель переобучаем после подбора; пороги переносим без подгонки на последней минуте.
 probability=predict(final_model,np.arange(N));np.save(OUT/f'{regime}_p_D.npy',probability)
 with (OUT/f'{regime}_model.pkl').open('wb') as f:pickle.dump(final_model,f)
 with (OUT/f'{regime}_model.pkl').open('rb') as f:loaded=pickle.load(f)
 assert np.allclose(predict(loaded,np.arange(0,N,503)),probability[::503],atol=1e-6)
 outputs={}
 for target in TARGETS:
  choice=winner['choices'][str(target)]
  predicted=np.zeros(N,bool)
  for _,a,b in BLOCKS:predicted[a:b+1]=filter_segment(probability[a:b+1]>=choice['threshold'],choice['min_dead_frames'])
  labels=np.where(graded,np.where(predicted,'D','LIVE'),'X')
  tag=f'{regime}_p{int(target*100)}'
  pd.DataFrame({'frame':np.arange(N),'label':labels}).to_csv(OUT/f'{tag}_predictions.csv',index=False)
  np.save(OUT/f'{tag}_dead_mask.npy',predicted)
  measurements={}
  for name,mask in [('public_5min',public),('last_minute',public&(positions>=START+6000))]:
   measurements[name]=score(truth[mask],predicted[mask])
   measurements[name]['D_average_precision']=float(average_precision_score(truth[mask],probability[mask]))
   measurements[name]['predicted_D_runs']=sum(bool(predicted[mask][a]) for a,b in spans(predicted[mask]))
  outputs[str(target)]=dict(selection=choice,metrics=measurements,prediction_file=f'{tag}_predictions.csv')
 all_results[regime]=dict(selected_config=winner['config'],candidates=candidates,outputs=outputs,
   train_samples=len(final_ids),train_D_samples=int(final_y.sum()),train_LIVE_samples=int((final_y==0).sum()),
   public_labels_used_for_training=regime in ['hybrid','public'],public_training_end_exclusive=START+6000-GUARD if regime in ['hybrid','public'] else None,
   selection_window='second-half weak labels' if regime=='weak' else 'public minute 4',
   evaluation_context='public minute 5: repeatedly viewed diagnostic, not independent; full offline filtering; causal features')
 save_json(OUT/'results.json',all_results)
pd.DataFrame(search_rows).to_csv(OUT/'development_threshold_search.csv',index=False)
save_json(OUT/'configuration.json',dict(experiment=EXPERIMENT,configs=CONFIGS,thresholds=THRESHOLDS,
 minimum_durations=MIN_DURATIONS,precision_targets=TARGETS,min_predicted_frames=MIN_PREDICTED,
 feature_names=feature_names,public_weight=1.0,guard_frames=GUARD,seed=42,
 weak_masks_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [align/'training_weak_D_mask.npy',align/'training_weak_LIVE_mask.npy']},
 features_sha256=hashlib.sha256(FEATURE_PATH.read_bytes()).hexdigest()))
progress(stage='completed',models=len(CONFIGS)+1)
print(json.dumps({k:{'selected':v['selected_config'],'outputs':v['outputs']} for k,v in all_results.items()},indent=2))
